In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Reading data file into our environment 

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/atharvbhandare16/teen-mental-health-dataset-v01/Teen_Mental_Health_Dataset.csv')
df.head()

In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv('/kaggle/input/datasets/atharvbhandare16/teen-mental-health-dataset-v01/Teen_Mental_Health_Dataset.csv')

# load into in-memory SQLite database
conn = sqlite3.connect(':memory:')
df.to_sql('mental_health', conn, index=False, if_exists='replace')

print("Table loaded successfully.")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
def sql(query):
    query_stripped = query.strip().upper()
    if query_stripped.startswith("SELECT") or query_stripped.startswith("WITH"):
        return pd.read_sql_query(query, conn)
    else:
        conn.execute(query)
        conn.commit()
        print("Query executed successfully.")

Feature Engineering 

In [ ]:
# create a temp table
sql("""
    create TEMP TABLE Data_prep_01 AS
    SELECT *,
    case when gender='male' then 1 else 0 end as gender_male_flag,
    case when gender='female' then 1 else 0 end as gender_female_flag,
    case when platform_usage in ('Instagram','Both') then 1 else 0 end as Instagram_user_flag,
    case when platform_usage in ('TikTok','Both') then 1 else 0 end as Tiktok_user_flag,
    CASE 
        WHEN social_interaction_level = 'low'    THEN 0
        WHEN social_interaction_level = 'medium' THEN 1
        WHEN social_interaction_level = 'high'   THEN 2
    END as social_interaction_encoded
    FROM mental_health

""")

Data EDA 

In [ ]:
import pandas as pd
import sqlite3

df = pd.read_sql_query("SELECT * FROM Data_prep_01", conn)

print("=" * 60)
print("1. SHAPE AND DATA TYPES")
print("=" * 60)
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)

print("\n" + "=" * 60)
print("2. FIRST 5 ROWS")
print("=" * 60)
display(df.head())

print("\n" + "=" * 60)
print("3. MISSING VALUES")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0]
if missing_df.empty:
    print("No missing values found.")
else:
    display(missing_df)

print("\n" + "=" * 60)
print("4. DUPLICATES")
print("=" * 60)
print("Total duplicate rows:", df.duplicated().sum())

print("\n" + "=" * 60)
print("5. TARGET VARIABLE DISTRIBUTION")
print("=" * 60)
print(df['depression_label'].value_counts())
print()
print(df['depression_label'].value_counts(normalize=True).mul(100).round(2).astype(str) + " %")

print("\n" + "=" * 60)
print("6. STATISTICAL SUMMARY")
print("=" * 60)
display(df.describe().T)

print("\n" + "=" * 60)
print("7. UNIQUE VALUES PER COLUMN")
print("=" * 60)
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique  |  sample: {df[col].unique()[:5]}")

print("\n" + "=" * 60)
print("8. CATEGORICAL COLUMNS — FULL VALUE COUNTS")
print("=" * 60)
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", cat_cols)
for col in cat_cols:
    print(f"\n── {col} ──")
    print(df[col].value_counts())

print("\n" + "=" * 60)
print("9. NUMERIC FEATURES — AVERAGE BY TARGET LABEL")
print("=" * 60)
exclude = ['depression_label', 'gender_male_flag', 'gender_female_flag',
           'Instagram_user_flag', 'Tiktok_user_flag']
num_cols = [c for c in df.select_dtypes(include=['int64','float64']).columns if c not in exclude]
display(df.groupby('depression_label')[num_cols].mean().round(2))

print("\n" + "=" * 60)
print("10. FLAG COLUMNS — DEPRESSION RATE WITHIN EACH FLAG")
print("=" * 60)
flag_cols = ['gender_male_flag', 'Instagram_user_flag', 'Tiktok_user_flag']
for col in flag_cols:
    total     = df[col].sum()
    depressed = df[df[col] == 1]['depression_label'].sum()
    pct       = round(depressed / total * 100, 2) if total > 0 else 0
    print(f"{col}: {int(total)} teens flagged → {int(depressed)} depressed ({pct}%)")

print("\n" + "=" * 60)
print("11. CORRELATION WITH TARGET")
print("=" * 60)
corr_cols = num_cols + flag_cols
corr = df[corr_cols + ['depression_label']].corr()['depression_label'].drop('depression_label')
print(corr.sort_values(ascending=False).round(3))

print("\n" + "=" * 60)
print("12. OUTLIER CHECK (IQR METHOD)")
print("=" * 60)
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
    print(f"{col}: {len(outliers)} outliers")

Dropping unnecessay Columns 

In [ ]:
# ── drop columns that are not needed ─────────────────────────────
drop_cols = [
    'User_id',                   # serial number
    'gender',                    # already encoded
    'platform_usage',            # already encoded
    'social_interaction_level',  # already encoded
    'gender_female_flag'         # dummy variable trap
]

df_model = df.drop(columns=drop_cols)

print("Remaining columns:", df_model.columns.tolist())
print("Shape:", df_model.shape)


Defining features and target variable 

In [ ]:
# ── define features and target ────────────────────────────────────
X = df_model.drop(columns=['depression_label'])
y = df_model['depression_label']

print("Features:", X.columns.tolist())
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nPositive class count:", y.sum())
print("Negative class count:", (y == 0).sum())
print("Scale pos weight to use:", round((y == 0).sum() / y.sum(), 1))

Train test split (80-20%)

In [ ]:
# ── train test split ──────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "| positives:", y_train.sum())
print("X_test :", X_test.shape,  "| positives:", y_test.sum())

Training XG Boost model 

In [ ]:
from xgboost import XGBClassifier
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

# ── base xgboost model ────────────────────────────────────────────
base_model = XGBClassifier(
    n_estimators   = 100,
    learning_rate  = 0.1,
    max_depth      = 4,
    subsample      = 0.8,
    scale_pos_weight = 38,
    random_state   = 42,
    eval_metric    = 'logloss',
    verbosity      = 0
)

# ── wrap in balanced bagging classifier ───────────────────────────
bbc = BalancedBaggingClassifier(
    estimator          = base_model,
    n_estimators       = 5,        # 5 buckets
    sampling_strategy  = 0.2,      # 1:5 ratio (pos:neg)
    replacement        = True,     # with replacement (bagging)
    random_state       = 42
)

# ── stratified kfold ──────────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── cross validation scores ───────────────────────────────────────
print("Running cross validation...")
print("(5 folds × 5 buckets = 25 models trained)\n")

recall_scores    = cross_val_score(bbc, X_train, y_train, cv=skf, scoring='recall')
precision_scores = cross_val_score(bbc, X_train, y_train, cv=skf, scoring='precision')
f1_scores        = cross_val_score(bbc, X_train, y_train, cv=skf, scoring='f1')

print("── Recall per fold    :", recall_scores.round(3))
print("── Mean Recall        :", recall_scores.mean().round(3))
print("── Std Recall         :", recall_scores.std().round(3))

print("\n── Precision per fold :", precision_scores.round(3))
print("── Mean Precision     :", precision_scores.mean().round(3))
print("── Std Precision      :", precision_scores.std().round(3))

print("\n── F1 per fold        :", f1_scores.round(3))
print("── Mean F1            :", f1_scores.mean().round(3))
print("── Std F1             :", f1_scores.std().round(3))

In [ ]:
# ── train final model on full training set ────────────────────────
print("Training final model on full X_train...")
bbc.fit(X_train, y_train)
print("Done.")

Evaluation metrics

In [ ]:
# ── evaluate on test set ──────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

y_test_pred  = bbc.predict(X_test)
y_test_proba = bbc.predict_proba(X_test)[:, 1]

print("=" * 50)
print("FINAL EVALUATION ON TEST SET")
print("=" * 50)
print(classification_report(y_test, y_test_pred))

cm = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix:")
print(f"                   Predicted 0   Predicted 1")
print(f"Actual 0 (healthy)     {cm[0][0]}           {cm[0][1]}")
print(f"Actual 1 (depressed)   {cm[1][0]}           {cm[1][1]}")

print("\nBreakdown:")
print(f"True  Negatives  (TN) - correctly predicted healthy   : {cm[0][0]}")
print(f"False Positives  (FP) - healthy flagged as depressed  : {cm[0][1]}")
print(f"False Negatives  (FN) - depressed teens missed        : {cm[1][0]}")
print(f"True  Positives  (TP) - depressed teens caught        : {cm[1][1]}")

In [ ]:
# ── threshold analysis ────────────────────────────────────────────
import pandas as pd

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
rows = []

for t in thresholds:
    y_pred = (y_test_proba >= t).astype(int)
    TP = int(((y_pred == 1) & (y_test == 1)).sum())
    TN = int(((y_pred == 0) & (y_test == 0)).sum())
    FP = int(((y_pred == 1) & (y_test == 0)).sum())
    FN = int(((y_pred == 0) & (y_test == 1)).sum())
    precision = round(TP / (TP + FP) * 100, 1) if (TP + FP) > 0 else 0
    recall    = round(TP / (TP + FN) * 100, 1) if (TP + FN) > 0 else 0
    f1        = round(2 * precision * recall / (precision + recall), 1) if (precision + recall) > 0 else 0
    rows.append({'Threshold': t, 'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
                 'Precision %': precision, 'Recall %': recall, 'F1 %': f1})

display(pd.DataFrame(rows))

In [ ]:
# ── evaluate on train and test at multiple thresholds ──────────────
import pandas as pd
import numpy as np
from sklearn.metrics import log_loss

y_train_proba = bbc.predict_proba(X_train)[:, 1]
y_test_proba  = bbc.predict_proba(X_test)[:, 1]

thresholds = [round(t, 1) for t in np.arange(0.1, 1.0, 0.1)]

def get_metrics_table(y_true, y_proba, dataset_name):
    rows = []
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        TP = int(((y_pred == 1) & (y_true == 1)).sum())
        TN = int(((y_pred == 0) & (y_true == 0)).sum())
        FP = int(((y_pred == 1) & (y_true == 0)).sum())
        FN = int(((y_pred == 0) & (y_true == 1)).sum())

        accuracy  = round((TP + TN) / (TP + TN + FP + FN) * 100, 1)
        precision = round(TP / (TP + FP) * 100, 1) if (TP + FP) > 0 else 0.0
        recall    = round(TP / (TP + FN) * 100, 1) if (TP + FN) > 0 else 0.0
        f1        = round(2 * precision * recall / (precision + recall), 1) if (precision + recall) > 0 else 0.0

        rows.append({
            'Threshold'  : t,
            'TP'         : TP,
            'TN'         : TN,
            'FP'         : FP,
            'FN'         : FN,
            'Accuracy %' : accuracy,
            'Precision %': precision,
            'Recall %'   : recall,
            'F1 %'       : f1
        })

    df_result = pd.DataFrame(rows)
    print(f"\n{'='*95}")
    print(f"{dataset_name} — THRESHOLD-WISE METRICS")
    print(f"{'='*95}")
    display(df_result)
    return df_result

train_results = get_metrics_table(y_train, y_train_proba, "TRAIN SET")
test_results  = get_metrics_table(y_test, y_test_proba, "TEST SET")

In [ ]:
# ── log loss ────────────────────────────────────────────────────────
train_logloss = log_loss(y_train, y_train_proba)
test_logloss  = log_loss(y_test, y_test_proba)

print("=" * 50)
print("LOG LOSS")
print("=" * 50)
print(f"Train Log Loss : {train_logloss:.4f}")
print(f"Test  Log Loss : {test_logloss:.4f}")
print(f"Gap            : {abs(train_logloss - test_logloss):.4f}")
print("\n(<0.2=excellent, 0.2-0.4=good, >0.6=poor)")